In [1]:
import pandas as pd
import numpy as np
import os
import json

import optuna
from optuna import Trial
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import kaggle

os.environ['KAGGLE_USERNAME'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['username']
os.environ['KAGGLE_KEY'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['key']


/home/osman/Desktop/Projects/ING_Datathon/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_data = pd.read_csv("train_data.csv")
test_data = pd.read_csv("test_data.csv")
sample_submission = pd.read_csv("sample_submission.csv")

In [ ]:
def recall_at_k(y_true, y_prob, k=0.1):
    """
    Tahmin edilen olasılıkların en üst k%'sını pozitif etiketleyerek recall değerini hesaplar.

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.
        k (float): Pozitif etiketlenecek olasılıkların yüzdelik dilimi (varsayılan 0.1).

    Döndürür:
        float: En iyi k% tahminlerindeki recall oranı.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    m = max(1, int(np.round(k * n)))
    order = np.argsort(-y_prob, kind="mergesort")
    top = order[:m]

    tp_at_k = y_true[top].sum()
    P = y_true.sum()

    return float(tp_at_k / P) if P > 0 else 0.0


def lift_at_k(y_true, y_prob, k=0.1):
    """
    Tahmin edilen olasılıkların en üst k%'sını pozitif etiketleyerek lift (precision/prevalence) değerini hesaplar.

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.
        k (float): Pozitif etiketlenecek olasılıkların yüzdelik dilimi (varsayılan 0.1).

    Döndürür:
        float: En iyi k% tahminlerindeki lift değeri.
    """
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)
    n = len(y_true)
    m = max(1, int(np.round(k * n)))
    order = np.argsort(-y_prob, kind="mergesort")
    top = order[:m]

    tp_at_k = y_true[top].sum()
    precision_at_k = tp_at_k / m
    prevalence = y_true.mean()

    return float(precision_at_k / prevalence) if prevalence > 0 else 0.0


def convert_auc_to_gini(auc):
    """
    ROC AUC skorunu Gini katsayısına dönüştürür.

    Gini katsayısı, ROC AUC skorunun doğrusal bir dönüşümüdür.

    Parametreler:
        auc (float): ROC AUC skoru (0 ile 1 arasında).

    Döndürür:
        float: Gini katsayısı (-1 ile 1 arasında).
    """
    return 2 * auc - 1


def ing_hubs_datathon_metric(y_true, y_prob):
    """
    Gini, recall@10% ve lift@10% metriklerini birleştiren özel bir metrik hesaplar.

    Metrik, her bir skoru bir baseline modelin metrik değerlerine göre oranlar ve aşağıdaki ağırlıkları uygular:
    - Gini: %40
    - Recall@10%: %30
    - Lift@10%: %30

    Parametreler:
        y_true (list): Gerçek ikili etiketler.
        y_prob (list): Tahmin edilen olasılıklar.

    Döndürür:
        float: Ağırlıklandırılmış bileşik skor.
    """
    # final metrik için ağırlıklar
    score_weights = {
        "gini": 0.4,
        "recall_at_10perc": 0.3,
        "lift_at_10perc": 0.3,
    }

    # baseline modelin her bir metrik için değerleri
    baseline_scores = {
        "roc_auc": 0.6925726757936908,
        "recall_at_10perc": 0.18469015795868773,
        "lift_at_10perc": 1.847159286784029,
    }

    # y_prob tahminleri için metriklerin hesaplanması
    roc_auc = roc_auc_score(y_true, y_prob)
    recall_at_10perc = recall_at_k(y_true, y_prob, k=0.1)
    lift_at_10perc = lift_at_k(y_true, y_prob, k=0.1)

    new_scores = {
        "roc_auc": roc_auc,
        "recall_at_10perc": recall_at_10perc,
        "lift_at_10perc": lift_at_10perc,
    }

    # roc auc değerlerinin gini değerine dönüştürülmesi
    baseline_scores["gini"] = convert_auc_to_gini(baseline_scores["roc_auc"])
    new_scores["gini"] = convert_auc_to_gini(new_scores["roc_auc"])

    # baseline modeline oranlama
    final_gini_score = new_scores["gini"] / baseline_scores["gini"]
    final_recall_score = new_scores["recall_at_10perc"] / baseline_scores["recall_at_10perc"]
    final_lift_score = new_scores["lift_at_10perc"] / baseline_scores["lift_at_10perc"]

    # ağırlıklandırılmış metriğin hesaplanması
    final_score = (
        final_gini_score * score_weights["gini"] +
        final_recall_score * score_weights["recall_at_10perc"] + 
        final_lift_score * score_weights["lift_at_10perc"]
    )
    return final_score

In [4]:
train_data

,age,tenure,cust_age_month,uses_mobile_eft,uses_cc,uses_any_digital_channel,mobile_eft_cnt_mean,mobile_eft_cnt_std,mobile_eft_cnt_min,mobile_eft_cnt_max,...,work_type_Unemployed,work_sector_Finance,work_sector_Healthcare,work_sector_Manufacturing,work_sector_Public Sector,work_sector_Retail,work_sector_Retired,work_sector_Student,work_sector_Technology,work_sector_Unemployed
0,64,135,633,1,0,1,2.238095,1.220851,1.0,5.0,...,0,0,0,0,0,0,0,0,1,0
1,22,47,217,1,1,1,1.676471,1.006662,1.0,4.0,...,0,0,0,0,0,0,0,1,0,0
2,27,108,216,1,1,1,2.555556,1.476309,1.0,6.0,...,0,1,0,0,0,0,0,0,0,0
3,40,187,293,1,1,1,7.142857,3.307839,4.0,14.0,...,1,0,0,0,0,0,0,0,0,1
4,64,218,550,1,1,1,0.793103,1.372675,0.0,5.0,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133282,54,217,431,1,1,1,1.393939,1.657170,0.0,6.0,...,0,0,0,0,1,0,0,0,0,0
133283,47,37,527,1,1,1,2.000000,1.174440,1.0,5.0,...,0,0,0,0,1,0,0,0,0,0
133284,66,227,565,1,1,1,9.055556,5.796277,1.0,22.0,...,0,0,0,0,0,0,1,0,0,0
133285,31,156,216,1,1,1,3.576923,1.836803,1.0,7.0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
X = train_data.drop("churn", axis=1)
y = train_data["churn"]

In [6]:
scaler = MinMaxScaler((0,1))

In [7]:
X[X.columns] = scaler.fit_transform(X)
test_data[test_data.columns] = scaler.transform(test_data)

In [8]:
def objective(trial: Trial, X: pd.DataFrame, y: np.ndarray, n_splits: int = 5) -> float:
    # Hyperparametreler
    params = {
        'objective': 'binary',
        'verbose': False,
        'metric': 'binary_logloss',
        'verbosity': -1,
        'random_state': 42,
        'boosting_type': 'gbdt',
        'num_leaves': trial.suggest_int('num_leaves', 30, 200),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.1, 10.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample_freq': trial.suggest_int('subsample_freq', 1, 10),
    }

    # Dengesizlik: Pozitif sınıf ağırlığını optimize et
    neg, pos = np.bincount(y)
    scale_pos_weight = neg / pos
    params['scale_pos_weight'] = trial.suggest_float('scale_pos_weight', scale_pos_weight * 0.5, scale_pos_weight * 2)

    # K-Fold
    kf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    custom_scores = []

    for train_idx, val_idx in kf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        

        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)]
        )

        y_pred_proba = model.predict_proba(X_val)[:, 1]

        # Özel metrik
        score = ing_hubs_datathon_metric(y_val, y_pred_proba)
        custom_scores.append(score)

    return np.mean(custom_scores)

In [9]:
# Optimize et
study = optuna.create_study(direction='maximize', study_name='lgbm-churn-ing-metric')
study.optimize(
    lambda trial: objective(trial, X, y),
    n_trials=50,
    show_progress_bar=True
)

print("Best trial score:", study.best_trial.value)
print("Best params:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

[I 2025-10-22 09:35:59,784] A new study created in memory with name: lgbm-churn-ing-metric
Best trial: 0. Best value: 1.14242:   2%|▏         | 1/50 [00:36<30:04, 36.82s/it]

[I 2025-10-22 09:36:36,604] Trial 0 finished with value: 1.1424188911285087 and parameters: {'num_leaves': 185, 'max_depth': 12, 'learning_rate': 0.02022872483367157, 'n_estimators': 460, 'subsample': 0.9308552592115025, 'colsample_bytree': 0.8997259245270901, 'reg_alpha': 2.85838224011813, 'reg_lambda': 3.1213969165379485, 'min_child_samples': 75, 'subsample_freq': 10, 'scale_pos_weight': 4.853210196007075}. Best is trial 0 with value: 1.1424188911285087.


Best trial: 1. Best value: 1.16709:   4%|▍         | 2/50 [01:02<24:04, 30.09s/it]

[I 2025-10-22 09:37:01,981] Trial 1 finished with value: 1.1670931277997156 and parameters: {'num_leaves': 151, 'max_depth': 11, 'learning_rate': 0.021685852519424138, 'n_estimators': 310, 'subsample': 0.7130289520960624, 'colsample_bytree': 0.9226061316870888, 'reg_alpha': 2.7943731842054027, 'reg_lambda': 2.3682114647324632, 'min_child_samples': 13, 'subsample_freq': 3, 'scale_pos_weight': 9.356044922033936}. Best is trial 1 with value: 1.1670931277997156.


Best trial: 1. Best value: 1.16709:   6%|▌         | 3/50 [01:13<16:58, 21.68s/it]

...

[I 2025-10-22 09:58:08,143] Trial 47 finished with value: 1.1819599938727707 and parameters: {'num_leaves': 45, 'max_depth': 6, 'learning_rate': 0.011337175796539409, 'n_estimators': 843, 'subsample': 0.6970982754076013, 'colsample_bytree': 0.8892036524436171, 'reg_alpha': 5.1914600743915145, 'reg_lambda': 8.994161521147078, 'min_child_samples': 14, 'subsample_freq': 6, 'scale_pos_weight': 5.197921560374406}. Best is trial 25 with value: 1.1854502747628266.


Best trial: 25. Best value: 1.18545:  98%|█████████▊| 49/50 [22:44<00:28, 28.94s/it]

[I 2025-10-22 09:58:44,123] Trial 48 finished with value: 1.0602289564843166 and parameters: {'num_leaves': 86, 'max_depth': 15, 'learning_rate': 0.06933584132281856, 'n_estimators': 766, 'subsample': 0.6459548071002794, 'colsample_bytree': 0.8583878878704511, 'reg_alpha': 4.489762784135266, 'reg_lambda': 8.604162430307955, 'min_child_samples': 23, 'subsample_freq': 8, 'scale_pos_weight': 5.856675946032773}. Best is trial 25 with value: 1.1854502747628266.


Best trial: 25. Best value: 1.18545: 100%|██████████| 50/50 [22:56<00:00, 27.52s/it]

[I 2025-10-22 09:58:55,972] Trial 49 finished with value: 1.1639272895814228 and parameters: {'num_leaves': 35, 'max_depth': 10, 'learning_rate': 0.011416480311751404, 'n_estimators': 295, 'subsample': 0.7548117598184865, 'colsample_bytree': 0.8078762862087455, 'reg_alpha': 4.054221219635232, 'reg_lambda': 9.558144100621547, 'min_child_samples': 33, 'subsample_freq': 1, 'scale_pos_weight': 7.153930802617387}. Best is trial 25 with value: 1.1854502747628266.
Best trial score: 1.1854502747628266
Best params:
  num_leaves: 30
  max_depth: 6
  learning_rate: 0.013222540938387022
  n_estimators: 790
  subsample: 0.8706752181315216
  colsample_bytree: 0.6570902425776276
  reg_alpha: 2.0006718616781405
  reg_lambda: 4.2216805970774836
  min_child_samples: 11
  subsample_freq: 3
  scale_pos_weight: 5.694097524061701


In [10]:
# En iyi parametreler
best_params = study.best_trial.params.copy()
best_params['objective'] = 'binary'
best_params['random_state'] = 42

# Final model
final_model = lgb.LGBMClassifier(**best_params)
final_model.fit(X, y)

# Tahmin
y_pred_proba = final_model.predict_proba(X)
# Tüm metrikleri yazdır
gini = convert_auc_to_gini(roc_auc_score(y, y_pred_proba[:, 1]))
recall_10 = recall_at_k(y, y_pred_proba, k=0.1)
lift_10 = lift_at_k(y, y_pred_proba, k=0.1)
final_score = ing_hubs_datathon_metric(y, y_pred_proba[:, 1])

print("\n📊 FINAL MODEL SKORLARI (TÜM VERİ ÜZERİNDE):")
print(f"Gini:              {gini:.4f}")
print(f"Recall@10%:        {recall_10:.4f}")
print(f"Lift@10%:          {lift_10:.4f}")
print(f"Final ING Metric:  {final_score:.4f}")


📊 FINAL MODEL SKORLARI (TÜM VERİ ÜZERİNDE):
Gini:              0.5310
Recall@10%:        0.0000
Lift@10%:          0.0000
Final ING Metric:  1.4384


In [11]:
cv_scores = []
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models:list[lgb.LGBMClassifier] = []
for train_idx, val_idx in kf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    model = lgb.LGBMClassifier(**best_params)
    model.fit(X_train, y_train)
    
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    score = ing_hubs_datathon_metric(y_val, y_pred_proba)
    cv_scores.append(score)
    models.append(model)

print(f"\n✅ 5-Fold CV ING Metric: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")


✅ 5-Fold CV ING Metric: 1.1855 ± 0.0159


In [12]:
test_predictions = np.mean([m.predict_proba(test_data)[:,1] for m in models], axis=0)

In [13]:
sample_submission["churn"] = test_predictions

In [14]:
sample_submission.to_csv('/tmp/submission.csv', index=False)
kaggle.api.competition_submit(
    file_name='/tmp/submission.csv', 
    message='lgbm with Optuna kfold and feature engineering history data ensemble kfold models', 
    competition='ing-hubs-turkiye-datathon'
)

100%|██████████| 1.06M/1.06M [00:01<00:00, 665kB/s] 


{"message": "Successfully submitted to ING Hubs T\u00fcrkiye Datathon", "ref": 47573885}